# **MultiPDF QA Retriever with FAISS and LangChain**

### **Objective:**  
 
In this demo, you will learn how to load multiple PDF research papers, split them into smaller chunks, create embeddings using OpenAI, store them in a FAISS vector database, and use a retriever to answer queries based on document content.

---

### **Note:**  
- Before running any demo, ensure that the **requirements.txt** file is installed. This file contains all the required dependencies for **all demos and guided practices under Building LLM Applications**.
- If the dependencies were already installed earlier (after creating the virtual environment), there is no need to install them again. You can directly proceed with running the demo.
- Refer to Lesson_01 **Demo_01_Zero_Shot_Prompting.ipynb** Step 1 for creating a virtual environment and installing the requirements.txt 
- Ensure you select the right kernel **Python (myenv)** while running the demos
---


### **Steps to perform:**
1. Import the necessary libraries  
2. Load and split the documents  
3. Load OpenAI embeddings  
4. Create and load the FAISS database  
5. Create and use the retriever  
6. Pass a query to the retriever  

---


### **Step 1: Importing the necessary libraries**

In [3]:


from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

/voc/work/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Step 2: Load and split the documents**


*   Create a directory named `GenAI_Papers`.
*   Load the PDF documents in the directory.
*   Split the documents into smaller chunks using the **RecursiveCharacterTextSplitter**.

In [4]:
# Loading the documents
doc_loader = DirectoryLoader('GenAI_Papers', glob="./*.pdf", loader_cls=PyPDFLoader)
documents = doc_loader.load()

# Splitting the documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)

texts

doc_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=64)
split_texts = doc_splitter.split_documents(texts)

print(len(split_texts))
split_texts


657


[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-02-10T01:38:06+00:00', 'author': '', 'keywords': '', 'moddate': '2023-02-10T01:38:06+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'GenAI_Papers/toolformer.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}, page_content='Toolformer: Language Models Can Teach Themselves to Use Tools\nTimo Schick Jane Dwivedi-Yu Roberto Dessì† Roberta Raileanu\nMaria Lomeli Luke Zettlemoyer Nicola Cancedda Thomas Scialom\nMeta AI Research †Universitat Pompeu Fabra\nAbstract\nLanguage models (LMs) exhibit remarkable\nabilities to solve new tasks from just a few\nexamples or textual instructions, especially at\nscale. They also, paradoxically, struggle with\nbasic functionality, such as arithmetic or fac-\ntual lookup, where much simpler and smaller\nmodels excel. In thi

### **Step 3: Load OpenAI embeddings**

In [5]:
openai_embeddings = OpenAIEmbeddings()


### **Step 4: Create and load the FAISS database**

- Create a FAISS vector database using embeddings
- Save it locally and load it back



In [6]:


# Create embeddings for texts
text_embeddings = openai_embeddings.embed_documents([text.page_content for text in texts])

# Creating the FAISS database
faiss_index = FAISS.from_texts([text.page_content for text in texts], openai_embeddings)

# Save the FAISS index
faiss_index.save_local('faiss_index')

# Load the FAISS index
faiss_index = FAISS.load_local(
    'faiss_index',
    openai_embeddings,
    allow_dangerous_deserialization=True
)


### **Step 5: Create and use the retriever**

*   Create a retriever using the vector database.
*   Use the retriever to get relevant documents for a specific query.



In [7]:
# Creating retriever
retriever = faiss_index.as_retriever()

# Using retriever
docs = retriever.invoke("What is toolformer?")


### **Step 6: Pass a query to the retriever**

- Pass a query to the vector database
- Print the most relevant document


In [8]:
query = "A fundamental limitation of HMMs"
docs = faiss_index.similarity_search(query)

print(docs[0].page_content)


with size, measured by the number of trainable parameters: f or example, W ei et al. (2022b) demonstrate
that LLMs become able to perform some BIG-bench tasks 3 via few-shot prompting once a certain scale is
attained. Although a recent line of work yielded smaller LMs that retain some capabilities from their largest
counterpart ( Hoﬀmann et al. , 2022), the size and need for data of LLMs can be impractical for tra ining
but also maintenance: continual learning for large models r emains an open research question ( Scialom et al. ,
2022). Other limitations of LLMs are discussed by Goldberg (2023) in the context of ChatGPT, a chatbot
built upon GPT3.
W e argue these issues stem from a fundamental defect of LLMs: they are generally trained to perform
statistical language modeling given (i) a single parametri c model and (ii) a limited context, typically the n
previous or surrounding tokens. While n has been growing in recent years thanks to software and hardw are


### **Conclusion**

By the end of this demo, you have a clear understanding of how to use LangChain’s MultiPDF retriever with FAISS. You’ve learned how to load and process documents, create a database, make a retriever, and use the retriever to ask questions. This knowledge will help you effectively utilize LangChain’s capabilities in your projects.

---